# Test CV Mapping, Gate Validation, HTML Rendering & PDF Export
This notebook runs the complete end-to-end mapping pipeline on a real structured CV.

In [ ]:
import sys
from pathlib import Path

backend_dir = (
    Path(".").resolve().parent
    if Path(".").resolve().name == "notebooks"
    else Path(".").resolve()
)
sys.path.insert(0, str(backend_dir))
from app.core.db import Database
from app.models.cv_tailoring import TailoredCVResponse, TailoringChangeItem
from app.services.cv_pipeline_persistence_service import persist_pipeline_tailoring
from app.services.cv_reconstruction_service import validate_reconstruction_gate
from app.services.cv_structuring_service import (
    build_manual_text_extraction,
    structure_cv,
)
from app.services.cv_template_render_service import render_cv_document
from app.services.tailored_cv_pdf import generate_tailored_cv_pdf

In [ ]:
cv_text = """NGUYEN THANH MINH DUY
AI Engineer / ML Engineer
ntminhduy123@gmail.com | +84 33-545-2060
Ho Chi Minh City, Vietnam
https://linkedin.com/in/minhduy-ai | https://github.com/zamminhduy123

SUMMARY
AI and machine learning researcher with a master's degree in PyTorch.

EDUCATION
Soonchunhyang University (SCH)
M.S. in Engineering (Fully Funded); GPA: 4.32/4.5
Mar 2024 – Feb 2026

EXPERIENCE
Zalo – VNG Corporation
Software Engineer, Zalo PC
May 2022 – Mar 2024
• Led frontend implementation for a full architectural revamp of the user Contact Card.

SKILLS
Programming Languages: Python, TypeScript, C++, SQL
Frameworks & Libraries: PyTorch, FastAPI, Next.js, Docker
"""

# Step 1: Structure CV
struct_res = await structure_cv(cv_text=cv_text)
doc = struct_res.document
print(
    f"Structured {len(doc.sections)} sections. Warnings: {doc.reconstruction_warnings}"
)
validate_reconstruction_gate(doc)
print("Reconstruction Gate: PASSED")

In [ ]:
# Step 2: Render Templates
for design in ["classic_ats", "modern_professional", "compact"]:
    res = render_cv_document(document=doc, template_id=design, language="vi")
    print(
        f"Design [{design}]: HTML size = {len(res.html)} bytes, Valid = {res.diagnostics.is_valid}"
    )

In [ ]:
# Step 3: Test Persistence & PDF Generation
user_row = await Database.fetch_one("SELECT id FROM users LIMIT 1")
user_id = user_row["id"]
canonical = doc.to_canonical_dict()
tailoring = TailoredCVResponse(
    tailored_cv=canonical,
    change_log=[
        TailoringChangeItem(
            path="experience[0].bullets[0].text",
            original_text=canonical["experience"][0]["bullets"][0]["text"],
            proposed_text="Architected frontend components for the user Contact Card revamp.",
            rationale="Enhanced action verbs while keeping core facts.",
        )
    ],
    tailoring_summary="Tailored experience bullets.",
)
raw_extraction = build_manual_text_extraction(cv_text)
version = await persist_pipeline_tailoring(
    user_id=user_id,
    source_text=cv_text,
    raw_extraction=raw_extraction,
    source_document=doc,
    job_description="AI Engineer",
    tailoring=tailoring,
    selected_design="classic_ats",
)
print(f"Version saved: {version.id}")
pdf_bytes = await generate_tailored_cv_pdf(
    tailored_cv=version.tailored_cv,
    document_v2=version.document_v2,
    design="classic_ats",
    language="vi",
)
print(f"PDF Generated: {len(pdf_bytes)} bytes")